# Fine-tune CodeT5+ and infer on the TypePro test split

Settings: **Internet ON**, accelerator **GPU**. Attach the final private
dataset `typepro-python-contrastive` using **Add Input**.


In [ ]:
REPOSITORY = 'https://github.com/duyvu1105/TypePro.git'
BRANCH = 'main'
MODEL_NAME = "Salesforce/codet5p-220m-py"
QUERY_LENGTH = 768
CANDIDATE_LENGTH = 256
EPOCHS = 3

import json
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/TypePro")
OUTPUT_DIR = Path("/kaggle/working/codet5p-typepro-python")

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run([str(value) for value in command], cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, REPO_DIR])
PIPELINE_DIR = REPO_DIR / "codet5p_type_retrieval"
run([sys.executable, "-m", "pip", "install", "-q", "-r", PIPELINE_DIR / "requirements.txt"])


## Locate and verify the attached processed dataset


In [ ]:
candidates = []
for path in Path("/kaggle/input").rglob("manifest.json"):
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if value.get("schema_version", "").startswith("typepro-codet5p-contrastive"):
        candidates.append(path.parent)
if len(candidates) != 1:
    raise RuntimeError(f"Expected exactly one TypePro processed dataset, found {candidates}")
DATA_DIR = candidates[0]
print("Using dataset:", DATA_DIR)
run([sys.executable, PIPELINE_DIR / "verify_dataset.py", "--data-dir", DATA_DIR])


## Contrastive fine-tuning


In [ ]:
run([
    sys.executable, "-u", PIPELINE_DIR / "train.py",
    "--data-dir", DATA_DIR,
    "--output-dir", OUTPUT_DIR,
    "--model-name", MODEL_NAME,
    "--projection-dim", 256,
    "--query-length", QUERY_LENGTH,
    "--candidate-length", CANDIDATE_LENGTH,
    "--batch-size", 2,
    "--gradient-accumulation-steps", 8,
    "--epochs", EPOCHS,
    "--learning-rate", "2e-5",
    "--mixed-precision", "fp16",
    "--gradient-checkpointing",
    "--preview-samples", 2,
    "--preview-max-chars", 1600,
    "--seed", 13,
])


## Batched inference and test metrics


In [ ]:
predictions = Path("/kaggle/working/test_predictions.jsonl")
run([
    sys.executable, "-u", PIPELINE_DIR / "infer.py",
    "--checkpoint", OUTPUT_DIR / "best",
    "--input", DATA_DIR / "test.jsonl",
    "--output", predictions,
    "--query-length", QUERY_LENGTH,
    "--candidate-length", CANDIDATE_LENGTH,
    "--batch-size", 4,
    "--top-k", 5,
    "--preview-samples", 3,
    "--log-every", 1000,
])
print("Predictions:", predictions)
